# Machine Learning Pipeline - Scoring New Data

Imaginemos que un colega del área de negocio nos pide hacer predicciones sobre datos nuevos. Veremos dos formas de hacerlo: primero aplicando cada transformación "a mano" como muchas veces se ve en la industria, y luego usando el pipeline entrenado y serializado para un enfoque limpio y reproducible.

---

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from scipy import stats

---

## 2. Cargar datos nuevos

En esta demo, simularemos datos nuevos usando el mismo test.csv pero el enfoque es idéntico si los datos vinieran de producción o sistemas.

In [ ]:
# Usamos el test set como "datos nuevos"
data_new = pd.read_csv('proyecto_pipeline_machine_learning_muestra/data/test.csv')
print(f"Shape de datos nuevos: {data_new.shape}")
data_new.head()

In [ ]:
# Eliminar columnas ID si existen
data_new = data_new.drop(columns=[c for c in data_new.columns if 'Id' in c or 'id' in c], errors='ignore')

---

## 3. MÉTODO 1 — SCORING MANUAL (Python puro)

Este es el enfoque "a mano", donde replicamos paso a paso todo el feature engineering y selección que hicimos sobre los datos de entrenamiento. Observa la cantidad de código, valores hardcodeados y potencial de errores.

### a) Castear variables categóricas disfrazadas de numéricas

In [ ]:
# Ejemplo: Si 'MSSubClass' es numérica pero debería ser categórica
if 'MSSubClass' in data_new.columns:
    data_new['MSSubClass'] = data_new['MSSubClass'].astype(str)

### b) Imputar categóricas: fillna("Missing") y fillna(moda)

Sabemos que imputamos Missing o moda para cada categórica.

- NOTA: Modas hardcodeadas obtenidas del train set (ejemplo de cómo se vería para algunas columnas comunes en House Prices)

In [ ]:
# Categóricas con 'Missing'
cat_fill_missing = ['Alley', 'PoolQC', 'Fence', 'MiscFeature', 'FireplaceQu']
for col in cat_fill_missing:
    if col in data_new.columns:
        data_new[col] = data_new[col].fillna('Missing')

# Otras categóricas con moda (ejemplo hardcodeado para MSZoning, KitchenQual...)
cat_fill_mode = {
    'MSZoning': 'RL',
    'KitchenQual': 'TA',
    'Exterior1st': 'VinylSd',
    'Exterior2nd': 'VinylSd',
    'SaleType': 'WD',
}
for col, moda in cat_fill_mode.items():
    if col in data_new.columns:
        data_new[col] = data_new[col].fillna(moda)

### c) Imputar numéricas: indicador binario + fillna(media)

Hardcodeamos las medias aprendidas del train:

In [ ]:
num_vars = ['LotFrontage', 'MasVnrArea', 'GarageYrBlt']  # ejemplo
def impute_num_with_indicator(df, col, mean):
    df[col + '_NA'] = np.where(df[col].isnull(), 1, 0)
    df[col] = df[col].fillna(mean)

num_means = {
    'LotFrontage': 70.0,   # ejemplo de media del train
    'MasVnrArea': 103.7,
    'GarageYrBlt': 1978.1,
}
for col, mean in num_means.items():
    if col in data_new.columns:
        impute_num_with_indicator(data_new, col, mean)

### d) Variables temporales: calcular años transcurridos, drop año referencia

In [ ]:
# Por ejemplo: 'YearRemodAdd' - 'YearBuilt' → 'YearsSinceRemodel'
if set(['YearRemodAdd','YearBuilt']).issubset(data_new.columns):
    data_new['YearsSinceRemodel'] = data_new['YearRemodAdd'] - data_new['YearBuilt']
    data_new = data_new.drop('YearRemodAdd', axis=1)

### e) Transformaciones: log, Yeo-Johnson (hardcodear lambda), binarizar

In [ ]:
# Aplicar log a ciertas variables (ejemplo: 'GrLivArea')
if 'GrLivArea' in data_new.columns:
    data_new['GrLivArea'] = np.log1p(data_new['GrLivArea'])

# Yeo-Johnson (lambda fijado en el valor aprendido del train)
# Suponiendo lambda de 0.15 para "TotalBsmtSF" extraído del entrenamiento
if 'TotalBsmtSF' in data_new.columns:
    data_new['TotalBsmtSF'] = stats.yeojohnson(data_new['TotalBsmtSF'], lmbda=0.15)

# Binarizar variable ejemplo:
if 'CentralAir' in data_new.columns:
    data_new['CentralAir'] = data_new['CentralAir'].map({'Y':1, 'N':0})

### f) Mappings ordinales: hardcodear diccionarios de calidad

In [ ]:
qual_maps = {
    'ExterQual': {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1},
    'ExterCond': {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1},
    'BsmtQual': {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1, 'Missing':0},
    # ...
}
for col, mapping in qual_maps.items():
    if col in data_new.columns:
        data_new[col] = data_new[col].map(mapping)

### g) Rare labels: hardcodear categorías frecuentes por variable

In [ ]:
# Ejemplo: sólo mantener top-5 en Neighborhood
if 'Neighborhood' in data_new.columns:
    frequent = ['NAmes', 'CollgCr', 'OldTown', 'Edwards', 'Somerst']
    data_new['Neighborhood'] = np.where(
        data_new['Neighborhood'].isin(frequent),
        data_new['Neighborhood'], 'Rare')

### h) Encoding ordinal: hardcodear mappings categoría→número

In [ ]:
ord_maps = {
    'BsmtExposure': {'Gd':4, 'Av':3, 'Mn':2, 'No':1, 'Missing':0},
    # ...
}
for col, mapping in ord_maps.items():
    if col in data_new.columns:
        data_new[col] = data_new[col].map(mapping)

### i) Verificar NaN restantes, fillna(0) como parche

In [ ]:
# Parche para no estrellar el modelo
if data_new.isnull().sum().sum() > 0:
    data_new = data_new.fillna(0)

### j) Feature scaling: cargar scaler serializado

In [ ]:
# Cargar scaler
scaler = joblib.load('proyecto_pipeline_machine_learning_muestra/scaler.pkl')
# Cuidado: puede lanzar error si el orden de columnas difiere
X_scaled = scaler.transform(data_new)

### k) Seleccionar features: cargar lista final de features

In [ ]:
selected_features = pd.read_csv('proyecto_pipeline_machine_learning_muestra/selected_features.csv')['feature'].tolist()
# Seleccionar sólo las features que el modelo espera (pueden faltar columnas)
from sklearn.utils.validation import check_is_fitted

# Si el scaler transforma todas y necesitamos sólo las seleccionadas:
features_df = pd.DataFrame(X_scaled, columns=data_new.columns) 
X_final = features_df[selected_features]

### l) Cargar modelo y predecir

In [ ]:
model = joblib.load('proyecto_pipeline_machine_learning_muestra/model.pkl')
y_pred_log = model.predict(X_final)

### m) Visualizar predicciones

In [ ]:
# Suponiendo problema de regresión con target transformado a log
preds = np.exp(y_pred_log)
plt.figure(figsize=(8,4))
plt.hist(preds, bins=30, color='steelblue', edgecolor='k')
plt.title('Distribución de Predicciones (scoring manual)')
plt.xlabel('Target predicho (original scale)')
plt.show()

---

**Problemas de este enfoque:**
- Mucho código repetitivo y difícil de mantener
- Valores hardcodeados que si cambian, pueden romper el scoring
- Se procesan variables incluso si el modelo no las utiliza
- Si los datos nuevos tienen NaNs inesperados, puede fallar silenciosamente

---

## 4. MÉTODO 2 — SCORING CON PIPELINE (recomendado)

Ahora repetimos el scoring, pero usando el pipeline serializado. Observa cuán limpio y robusto es el código.

**El pipeline fue entrenado para encapsular _todas_ las transformaciones y pasos de selección.**

In [ ]:
# Volvemos a cargar los datos nuevos para empezar 100% fresco
X_unseen = pd.read_csv('proyecto_pipeline_machine_learning_muestra/data/test.csv')
X_unseen = X_unseen.drop(columns=[c for c in X_unseen.columns if 'Id' in c or 'id' in c], errors='ignore')

# Asegurarnos (si aplica) de castear categóricas disfrazadas de numéricas
if 'MSSubClass' in X_unseen.columns:
    X_unseen['MSSubClass'] = X_unseen['MSSubClass'].astype(str)

### a) Cargar pipeline completo

In [ ]:
pipeline = joblib.load('proyecto_pipeline_machine_learning_muestra/preprocessors.pkl')

### b) Transformar datos nuevos

In [ ]:
X_processed = pipeline.transform(X_unseen)

### c) Seleccionar features finales

In [ ]:
selected_features = pd.read_csv('proyecto_pipeline_machine_learning_muestra/selected_features.csv')['feature'].tolist()

# Puede ser necesario convertir a DataFrame dependiendo del pipeline
if not isinstance(X_processed, pd.DataFrame):
    X_processed = pd.DataFrame(X_processed, columns=pipeline.get_feature_names_out())

# Seleccionar sólo features del modelo
X_final = X_processed[selected_features]

### d) Scoring

In [ ]:
model = joblib.load('proyecto_pipeline_machine_learning_muestra/model.pkl')
y_pred_log = model.predict(X_final)
y_pred = np.exp(y_pred_log)

### e) Visualización

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(y_pred, bins=30, color='seagreen', edgecolor='k')
plt.title('Distribución de Predicciones (pipeline)')
plt.xlabel('Target predicho (original scale)')
plt.show()

---

**Beneficios del pipeline:**
- El mismo resultado con muchísimo menos código, sin repeticiones ni "copy-paste"
- No hay hardcoding de modas, medias, mappings, lmbdas, ni manejo manual de NaN
- Sólo las features que el modelo necesita llegan al scoring
- Más fácil de mantener, versionar y poner en producción

---

## Conclusiones

- El Pipeline de Machine Learning encapsula todas las transformaciones aprendidas durante el entrenamiento.
- No hay riesgo de olvidar pasos, hardcodear valores incorrectos o romper el flujo ante cambios futuros.
- El método recomendado para scoring ML en producción es SIEMPRE usar el pipeline serializado y versionado.
- Menos código, menos bugs, mejores resultados - y tus modelos serán fácilmente auditables y reproducibles.